### Introduction To Data Ingestion

In [3]:
import langchain

In [4]:
import os 
from typing import List, Dict, Any
import pandas as pd

In [5]:
from langchain_core.documents import Document
from langchain_text_splitters import (RecursiveCharacterTextSplitter, CharacterTextSplitter, TokenTextSplitter)
print("Setup complete")

Setup complete


#Understanding Document Structure In Langchain

In [8]:
### creat a simple document
doc = Document(
    page_content="This is a simple document. It contains some text to demonstrate the Document class in LangChain.", 
    metadata={
        "source": "example.txt",
        "page": 1,
        "author": "John Doe",
        "date_created": "2024-06-01",
        "custom_field": "custom_value"
        })

print("Document Structure:")
print(f"Page Content: {doc.page_content}")
print(f"Metadata: {doc.metadata}")

Document Structure:
Page Content: This is a simple document. It contains some text to demonstrate the Document class in LangChain.
Metadata: {'source': 'example.txt', 'page': 1, 'author': 'John Doe', 'date_created': '2024-06-01', 'custom_field': 'custom_value'}


### Text Files (.txt) - The simplest Case {#2-text-files}

In [9]:
### Create a simple txt file
import os
os.makedirs("data/text_files", exist_ok=True)

In [ ]:
sample_text = {
    "data/text_files/python_intro.txt": 
    """Python is a high-level, interpreted programming language that is widely used for web development, data analysis, artificial intelligence,
    and scientific computing. It was created by Guido van Rossum and first released in 1991. Python's design philosophy emphasizes code readability and simplicity, making it an 
    excellent choice for beginners and experienced developers alike.""",

    "data/text_files/rag_explanation.txt":
    """Retrieval-Augmented Generation (RAG) is a technique used to give Large Language Models (LLMs) access to specific, private data. 
    It works by retrieving relevant documents from a vector database and "stuffing" them into the prompt as context. 
    This reduces hallucinations because the model bases its answer on the provided facts rather than its training data alone.""",

    "data/text_files/vector_db.txt":
    """A vector database, like FAISS or ChromaDB, stores data as numerical representations called embeddings. 
    These embeddings capture the "meaning" of the text. When a user asks a question, the database performs a similarity search 
    to find pieces of text that are mathematically close to the query, even if they don't share the exact same words.""",

    "data/text_files/ai_history.txt":
    """The field of Artificial Intelligence began in the mid-20th century. Alan Turing's 1950 paper "Computing Machinery and Intelligence" 
    proposed the Turing Test as a measure of machine intelligence. Since then, the field has evolved from simple rule-based systems 
    to the complex neural networks and transformers that power modern generative AI today."""
}

for path, content in sample_text.items():
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)
        print(f"Created: {path}")

### TextLoader - Read Single file

In [ ]:
from langchain_community.document_loaders import TextLoader

### loading a single text file.
loader= TextLoader("data/text_files/ai_history.txt", encoding="utf-8")
documents= loader.load()
print(f"Loaded {len(documents)} document")
print (f"Content preview: {documents[0].page_content[:100]}...")
print(f"Metadata: {documents[0].metadata}")

#### DirectoryLoader - Multiple Text Files

In [ ]:
from langchain_community.document_loaders import DirectoryLoader

### load all the text files form the directory.
dir_loader= DirectoryLoader(
    "data/text_files", #directory path
    glob="**/*.txt", #Pattern to match files
    loader_cls= TextLoader, #loader class to use
    loader_kwargs={'encoding':'utf-8'},
    show_progress= True
)
documents = dir_loader.load() 

print(f"Loaded {len(documents)} document")
for i , doc in enumerate(documents):
    print(f"\nDocument {i+1}:")
    print (f" Source: {doc.metadata['source']}")
    print(f"Lenght: {len(doc.page_content)} characters")


# DirectoryLoader characteristics

    # Advantages
    # loades Multiple files
    # Supports glob patterns
    # Progress Tracking
    # Recursive directory scanning

    # Disadvantages
    # All files must be the same type 
    # Limited error handling per file
    # Can be memory intensive for large directories
    



### Text Splitting Stratergies

In [10]:
## Different text splitting strategies
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter
)

print(documents)

[Document(metadata={'source': 'data\\text_files\\ai_history.txt'}, page_content='The field of Artificial Intelligence began in the mid-20th century. Alan Turing\'s 1950 paper "Computing Machinery and Intelligence" \n    proposed the Turing Test as a measure of machine intelligence. Since then, the field has evolved from simple rule-based systems \n    to the complex neural networks and transformers that power modern generative AI today.'), Document(metadata={'source': 'data\\text_files\\python_intro.txt'}, page_content="Python is a high-level, interpreted programming language that is widely used for web development, data analysis, artificial intelligence,\n    and scientific computing. It was created by Guido van Rossum and first released in 1991. Python's design philosophy emphasizes code readability and simplicity, making it an \n    excellent choice for beginners and experienced developers alike."), Document(metadata={'source': 'data\\text_files\\rag_explanation.txt'}, page_content=

In [11]:
### Method 1 - Character Text Splitter

text= documents[0].page_content
text

'The field of Artificial Intelligence began in the mid-20th century. Alan Turing\'s 1950 paper "Computing Machinery and Intelligence" \n    proposed the Turing Test as a measure of machine intelligence. Since then, the field has evolved from simple rule-based systems \n    to the complex neural networks and transformers that power modern generative AI today.'

In [14]:
# Method 1: Character-based splitting
print(" CHARACTER TEXT SPLITTER")
char_splitter= CharacterTextSplitter(
    separator= "\n", #Split on newlines
    chunk_size= 200, #Max chuck size in characters
    chunk_overlap= 20, #overlap between chucks
    length_function= len #How to measure chuck size
)

#we use split text because we have text not documents here
char_chunks=char_splitter.split_text(text)

print(f"Created {len(char_chunks)} chunks")
print(f"First chunk: {char_chunks[0][:100]}...")


 CHARACTER TEXT SPLITTER
Created 3 chunks
First chunk: The field of Artificial Intelligence began in the mid-20th century. Alan Turing's 1950 paper "Comput...


In [15]:
print(char_chunks[0])
print("-----------------")
print(char_chunks[1])

The field of Artificial Intelligence began in the mid-20th century. Alan Turing's 1950 paper "Computing Machinery and Intelligence"
-----------------
proposed the Turing Test as a measure of machine intelligence. Since then, the field has evolved from simple rule-based systems


In [ ]:
# Method 1: Recursive character splitting (Reccomended)
print("RECURSIVE CHARACTER TEXT SPLITTER")
recursive_splitter= RecursiveCharacterTextSplitter(
    separators=["\n\n","\n"," ",""], #Split on newlines
    chunk_size= 200, #Max chuck size in characters
    chunk_overlap= 20, #overlap between chucks. take a small portion of first chunk put in second
    length_function= len #How to measure chuck size
)

recursive_chunks=recursive_splitter.split_text(text)

print(f"Created {len(recursive_chunks)} chunks")
print(f"First chunk: {recursive_chunks[0][:100]}...")

RECURSIVE CHARACTER TEXT SPLITTER
Created 3 chunks
First chunk: The field of Artificial Intelligence began in the mid-20th century. Alan Turing's 1950 paper "Comput...


In [19]:
print(recursive_chunks[0])
print("-----------------")
print(recursive_chunks[1])

The field of Artificial Intelligence began in the mid-20th century. Alan Turing's 1950 paper "Computing Machinery and Intelligence"
-----------------
proposed the Turing Test as a measure of machine intelligence. Since then, the field has evolved from simple rule-based systems


In [ ]:
#Text Splitting Comparison

# CharacterTextSplitter
# simple and predictable
# good for structured text
# may break mid-sentence (con)
# use when: text has clear delimiters

#RecursiveCharacterTextSplitter
# respects text structure
# tries multiple separators
# best general purpose splitter
# slightly complex (con)
# use when default choice 

# TokenTextSplitter
# respects model token limits
# more accurate
# slower than charcter based (cons)
# use with token limited models